In [ ]:
"""
MediAegis - Sistema de Diagnóstico Médico Asistido mediante Redes Bayesianas
Módulo 2: Redes Bayesianas - Inteligencia Artificial (SCC-1012)
"""

!pip install pgmpy networkx matplotlib pandas numpyimport matplotlib.pyplot as plt
import networkx as nx
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

def construir_red_bayesiama_mediaegis():
    # 1. Definir la estructura del DAG (Directed Acyclic Graph)
    # Dependencias: Fiebre -> Enfermedad, Tos -> Enfermedad, Dolor -> Enfermedad
    modelo = DiscreteBayesianNetwork([
        ('Fiebre', 'Enfermedad'),
        ('Tos', 'Enfermedad'),
        ('Dolor', 'Enfermedad')
    ])

    # 2. Definir Tablas de Probabilidad Condicional (CPDs)
    # Probabilidades a Priori de los Síntomas (0: No, 1: Sí)
    cpd_fiebre = TabularCPD(
        variable='Fiebre',
        variable_card=2,
        values=[[0.7], [0.3]],
        state_names={'Fiebre': ['No', 'Sí']}
    )

    cpd_tos = TabularCPD(
        variable='Tos',
        variable_card=2,
        values=[[0.6], [0.4]],
        state_names={'Tos': ['No', 'Sí']}
    )

    cpd_dolor = TabularCPD(
        variable='Dolor',
        variable_card=2,
        values=[[0.8], [0.2]],
        state_names={'Dolor': ['No', 'Sí']}
    )

    # CPD Condicional de la variable Enfermedad dado Fiebre, Tos y Dolor
    # Orden de combinaciones padres (Fiebre, Tos, Dolor):
    # (0,0,0), (0,0,1), (0,1,0), (0,1,1), (1,0,0), (1,0,1), (1,1,0), (1,1,1)
    p_enfermedad_si = [0.01, 0.15, 0.25, 0.50, 0.40, 0.70, 0.85, 0.95]
    p_enfermedad_no = [1.0 - p for p in p_enfermedad_si]

    cpd_enfermedad = TabularCPD(
        variable='Enfermedad',
        variable_card=2,
        values=[p_enfermedad_no, p_enfermedad_si],
        evidence=['Fiebre', 'Tos', 'Dolor'],
        evidence_card=[2, 2, 2],
        state_names={
            'Enfermedad': ['No', 'Sí'],
            'Fiebre': ['No', 'Sí'],
            'Tos': ['No', 'Sí'],
            'Dolor': ['No', 'Sí']
        }
    )

    # 3. Asignar las CPDs al modelo y validar la consistencia
    modelo.add_cpds(cpd_fiebre, cpd_tos, cpd_dolor, cpd_enfermedad)
    assert modelo.check_model(), "El modelo contiene errores estocásticos o de dimensión."
    print("✓ Modelo de Red Bayesiana 'MediAegis' construido y validado exitosamente.\n")

    return modelo

def ejecutar_consultas_inferencia(modelo):
    inferencia = VariableElimination(modelo)

    print("==================================================")
    print("  CONSULTAS DE INFERENCIA MÉRICA - MEDIAEGIS")
    print("==================================================\n")

    # Consulta 1: P(Enfermedad | Fiebre = Sí, Tos = Sí)
    q1 = inferencia.query(variables=['Enfermedad'], evidence={'Fiebre': 'Sí', 'Tos': 'Sí'})
    print("Consulta 1: P(Enfermedad | Fiebre = Sí, Tos = Sí)")
    print(q1)

    # Consulta 2: P(Enfermedad | Dolor = Sí)
    q2 = inferencia.query(variables=['Enfermedad'], evidence={'Dolor': 'Sí'})
    print("\nConsulta 2: P(Enfermedad | Dolor = Sí)")
    print(q2)

def graficar_dag(modelo):
    pos = {
        'Fiebre': (-1, 1),
        'Tos': (0, 1),
        'Dolor': (1, 1),
        'Enfermedad': (0, 0)
    }
    plt.figure(figsize=(8, 5))
    nx.draw(
        modelo, pos, with_labels=True, node_size=3500, node_color='lightblue',
        font_size=12, font_weight='bold', arrowsize=20, edge_color='gray'
    )
    plt.title("DAG de MediAegis: Diagnóstico Médico Asistido")
    plt.savefig("dag_mediaegis.png", bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    red_mediaegis = construir_red_bayesiama_mediaegis()
    ejecutar_consultas_inferencia(red_mediaegis)
    # graficar_dag(red_mediaegis) # Descomentar para visualizar y guardar gráfico